In [ ]:
import pandas as pd
import numpy as np
import re

try:
    from unidecode import unidecode
except ModuleNotFoundError:
    !pip install unidecode
    from unidecode import unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 5.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from google.colab import files

# Upload the two CSV files:
# 1. codification_data.csv
# 2. icd_d_p_pairs.csv
uploaded = files.upload()

print("Uploaded files:")
for name in uploaded.keys():
    print(name)

train_file = None
icd_ref_file = None

for name in uploaded.keys():
    lower = name.lower()

    if "codification" in lower:
        train_file = name

    elif "icd_d_p_pairs" in lower or "pairs" in lower:
        icd_ref_file = name

print("Detected training file:", train_file)
print("Detected ICD reference file:", icd_ref_file)

if train_file is None:
    raise ValueError("Could not detect codification_data.csv. Please upload the training file.")

if icd_ref_file is None:
    raise ValueError("Could not detect icd_d_p_pairs.csv. Please upload the ICD reference file.")

train_df = pd.read_csv(train_file)
icd_ref = pd.read_csv(icd_ref_file)

print("train_df shape:", train_df.shape)
print("icd_ref shape:", icd_ref.shape)

display(train_df.head())
display(icd_ref.head())

Saving codification_data (2).csv to codification_data (2).csv
Saving icd_d_p_pairs (1).csv to icd_d_p_pairs (1).csv
Uploaded files:
codification_data (2).csv
icd_d_p_pairs (1).csv
Detected training file: codification_data (2).csv
Detected ICD reference file: icd_d_p_pairs (1).csv
train_df shape: (13700, 2)
icd_ref shape: (179742, 3)


,Code,Literal
0,J9809,Hiperreactividad bronquial
1,J9801,broncoespástica
2,I420,miocardiopatía dilatada
3,Y831,HTA irc 6
4,R5600,Crisis febriles atípicas


,Code,D_P,Description
0,A00,D,Cólera
1,A000,D,"Cólera debido a Vibrio cholerae 01, biotipo ch..."
2,A001,D,"Cólera debido a Vibrio cholerae 01, biotipo El..."
3,A009,D,"Cólera, no especificado"
4,A01,D,Fiebres tifoidea y paratifoidea


In [ ]:
def clean_code(code):
    code = str(code).strip().upper()
    code = re.sub(r"\s+", "", code)
    return code

def basic_normalise_text(text):
    text = str(text)
    text = re.sub(r"<.*?>", " ", text)
    text = text.replace("´", "'").replace("`", "'").replace("’", "'")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_for_matching(text):
    text = str(text).lower()
    text = unidecode(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z0-9\s+/.-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
train_df["Code_clean"] = train_df["Code"].apply(clean_code)
icd_ref["Code_clean"] = icd_ref["Code"].apply(clean_code)

train_df["Literal_basic"] = train_df["Literal"].apply(basic_normalise_text)
train_df["Literal_match"] = train_df["Literal_basic"].apply(normalize_for_matching)

icd_ref["Description_basic"] = icd_ref["Description"].apply(basic_normalise_text)

valid_reference_codes = set(icd_ref["Code_clean"])
train_df["in_icd_reference"] = train_df["Code_clean"].isin(valid_reference_codes)

icd_ref_small = icd_ref[["Code_clean", "D_P", "Description"]].drop_duplicates("Code_clean")

train_df = train_df.merge(
    icd_ref_small,
    on="Code_clean",
    how="left"
)

icd10_df = train_df[train_df["in_icd_reference"] == True].copy()
flagged_icd9_df = train_df[train_df["in_icd_reference"] == False].copy()

icd10_df["y_category"] = icd10_df["Code_clean"].astype(str).str[0]
icd10_df["y_full_code"] = icd10_df["Code_clean"].astype(str)

print("ICD-10 rows:", icd10_df.shape)
print("Flagged ICD-9 rows:", flagged_icd9_df.shape)

print(icd10_df["y_category"].value_counts().sort_index())

ICD-10 rows: (9943, 10)
Flagged ICD-9 rows: (3757, 8)
y_category
0    1046
1     129
2       6
3     338
4      34
5      14
6       1
8      11
A      22
B     579
C     205
D     336
E     433
F     302
G     214
H     146
I     370
J     234
K     381
L     110
M     288
N     536
O    1495
P     101
Q     140
R     357
S      71
T      70
U      22
V     188
W       7
X      10
Y      36
Z    1711
Name: count, dtype: int64


In [ ]:
import os

os.makedirs("outputs", exist_ok=True)

train_df.to_csv("outputs/processed_all_codes_icd10_and_flagged_icd9.csv", index=False)
icd10_df.to_csv("outputs/processed_icd10_reference_valid_only.csv", index=False)

print("Saved processed files.")

Saved processed files.
